# Using OpenAI Python Library
Examples for non-streaming and streaming calls.


## Setup

In [ ]:
# imports
from dotenv import load_dotenv
load_dotenv(override=True)

# third-party libraries
from openai import OpenAI

# local helper
from streaming_utils import get_chunk_content

# Create reusable clients at module scope for OpenAI, Gemini, and Ollama
openai_client = OpenAI()

# Gemini uses a Google generative language-compatible base_url
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = OpenAI(base_url=gemini_url)

# Ollama local server
ollama_url = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=ollama_url)

def get_llm_instance(model):
    """Return the appropriate OpenAI-compatible client for the given model string."""
    if "gpt" in model or "o1" in model or "o3" in model:
        return openai_client
    if "gemini" in model:
        return gemini_client
    if ":" in model:  # local-style name (e.g. ollama)
        return ollama_client
    raise ValueError(f"Unknown model type for: {model}")

In [ ]:
# model
model_name = "gpt-4o"

# prepare message
system_message = "You are a helpful assistant."
user_message = "Tell me a joke about programming."

# OpenAI client expects a list of dicts
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]


## Non-Streaming

In [ ]:
def message(messages, model, **kwargs):
    used_kwargs = dict(kwargs)

    print("\nResponse: ")
    client = get_llm_instance(model)
    response = client.chat.completions.create(model=model, messages=messages, **used_kwargs)
    return response.choices[0].message.content


# simple test call
print(f"Using model: {model_name}")
response = message(messages, model_name)
print(response)


## Streaming

In [ ]:
def stream_message(messages, model_name, **kwargs):
    used_kwargs = dict(kwargs)

    client = get_llm_instance(model_name)
    stream = client.chat.completions.create(model=model_name, messages=messages, stream=True, **used_kwargs)
 
    full_response = ""
    for chunk in stream:
        # Prefer structured extraction; repr-based fallback is disabled by default
        content = get_chunk_content(chunk)
        if content:
            print(content, end="", flush=True)
            full_response += content
    return full_response


# simple test call
print(f"Using model: {model_name}")
_ = stream_message(messages, model_name)
